<a href="https://colab.research.google.com/github/Manar1433/DrDos-DNSDetection-Attack-/blob/main/Yet_another_copy_of_Malware_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("Data.csv")

In [ ]:
df.head()

In [ ]:
df.isna().sum().sum()

In [ ]:
df.fillna(0,inplace=True)

In [ ]:
df["class"].unique()

In [ ]:
df["class"] = df["class"].replace({"BENIGN":0,"MALWARE":1})

In [ ]:
df["class"].value_counts()

In [ ]:
plt.figure(figsize=(10,10))
df["class"].value_counts().plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


In [ ]:
X=df[["e_cblp","e_cp","e_cparhdr","e_maxalloc","e_sp","e_lfanew","NumberOfSections","CreationYear","FH_char0","FH_char1","sus_sections","non_sus_sections","packer","packer_type","E_text","E_data","filesize","E_file","fileinfo"]]
y=df[["class"]]

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X = X.select_dtypes(include=['number'])

In [ ]:
X = X.select_dtypes(include=['number'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import accuracy_score
y_pred=model.predict(X_test)
accuracy_score(y_test,y_pred)

In [ ]:
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test,y_pred)

In [ ]:
depths =[]
train_acc = []
test_acc = []

for d in range (1,13):

  model = RandomForestClassifier(max_depth=d)

  model.fit(X_train,y_train)

  y_pred_test = accuracy_score(y_test,model.predict(X_test))
  test_acc.append(y_pred_test)

  y_pred_train = accuracy_score(y_train,model.predict(X_train))
  train_acc.append(y_pred_train)


  depths.append(d)


  print(d,y_pred_train,y_pred_test)

In [ ]:
plt.plot(depths, train_acc, label="train")

plt.plot(depths, test_acc, label="test")

plt.xlabel("Max depths")

plt.ylabel("Accuracy")

plt.legend()

plt.show()

In [ ]:
best_depth=depths[test_acc.index(max(test_acc))]

print(best_depth)

In [ ]:
best_model = RandomForestClassifier(max_depth=best_depth)

best_model.fit(X_train,y_train)

In [ ]:
prediction=model.predict(X_test)
print(accuracy_score(y_test,prediction))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test,prediction))

In [ ]:
!pip install gradio

In [ ]:
!pip install pefile

In [ ]:
import gradio as gr

In [ ]:
import requests
import io
def extract_pe_features(url):
  file_content = io.BytesIO(requests.get(url).content)
  pe = pefile.PE(data=file_content.read())
  features=[]
  features.append(pe.FILE_HEADER.Machine)
  features.append(pe .FILE_HEADER.SizeOfOptionalHeader)
  features.append(pe.FILE_HEADER.Characteristics)
  features.append(pe.OPTIONAL_HEADER.MajorLinkerVersion)
  features.append(pe.OPTIONAL_HEADER.MinorLinkerVersion)
  features.append(pe.OPTIONAL_HEADER.SizeOfCode)
  features.append(pe.OPTIONAL_HEADER.SizeOfInitializedData)
  features.append(pe.OPTIONAL_HEADER.SizeOfUninitializedData)
  features.append(pe.OPTIONAL_HEADER.AddressOfEntryPoint)
  features.append(pe.OPTIONAL_HEADER.BaseOfCode)
  features.append(pe.OPTIONAL_HEADER.BaseOfData)
  features.append(pe.OPTIONAL_HEADER.ImageBase)
  features.append(pe.OPTIONAL_HEADER.SectionAlignment)
  features.append(pe.OPTIONAL_HEADER.FileAlignment)
  features.append(pe.OPTIONAL_HEADER.MajorOperatingSystemVersion)
  features.append(pe.OPTIONAL_HEADER.MinorOperatingSystemVersion)
  features.append(pe.OPTIONAL_HEADER.MajorImageVersion)
  features.append(pe.OPTIONAL_HEADER.MinorImageVersion)
  features.append(pe.OPTIONAL_HEADER.MajorSubsystemVersion)
  features.append(pe.OPTIONAL_HEADER.MinorSubsystemVersion)
  features.append(pe.OPTIONAL_HEADER.SizeOfImage)
  features.append(pe.OPTIONAL_HEADER.SizeOfHeaders)
  features.append(pe.OPTIONAL_HEADER.CheckSum)
  features.append(pe.OPTIONAL_HEADER.Subsystem)
  features.append(pe.OPTIONAL_HEADER.DllCharacteristics)
  features.append(pe.OPTIONAL_HEADER.SizeOfStackReserve)
  features.append(pe.OPTIONAL_HEADER.SizeOfStackCommit)
  features.append(pe.OPTIONAL_HEADER.SizeOfHeapReserve)
  features.append(pe.OPTIONAL_HEADER.SizeOfHeapCommit)
  features.append(pe.OPTIONAL_HEADER.LoaderFlags)
  features.append(pe.OPTIONAL_HEADER.NumberOfRvaAndSizes)
  features.append(len(pe.sections))
  imports = set()
  for entry in pe.DIRECTORY_ENTRY_IMPORT:
    for imp in entry.imports:
      imports.add(imp.name)
  features.append(len(imports))
  prediction = model.predict([features])
  if prediction[0]==0:
    return "Benign"
  else:
    return "Malware"

In [ ]:
from google.colab import files

In [ ]:
uploaded= files.upload()

In [ ]:
import gradio as gr
interface = gr.Interface(
fn=extract_pe_features,
inputs=[gr.File(label="Upload File")],
outputs="text",
title="Malware Detection")
interface.launch()